In [ ]:
# 필요 라이브러리 설치
!pip install statsmodels

# 코랩에서 한글 폰트 사용을 위한 설정
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 런타임 다시 시작 안내
# 위 코드 실행 후 상단 메뉴에서 [런타임] > [런타임 다시 시작]을 눌러주세요.
# 런타임을 다시 시작해야 한글 폰트가 적용됩니다.

In [ ]:
# 런타임 다시 시작 후, 이 셀을 실행하여 라이브러리와 폰트를 로드합니다.
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import norm
# 한글 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

# Z검정

- 한 회사에서 고객 만족도 설문조사를 시행. 이전 설문조사 결과 평균 만족도가 8점(𝜇_0=8), 표준 편차는 1.5점(𝜎 = 1.5) 알려져있음.  새로운 서비스 도입 후 100명의 고객의 대상으로 표본 조사를 했더니 평균 만족도가 8.2점(𝑥 ̅ = 8.2). 새로운 서비스 정책이 고객의 만족도를 유의미하게 향상시켰는가?


- Z-score의 계산:  $\frac{\bar{x} - \mu_0}{\sigma /\sqrt(n)}$
- Z=1.33의 계산: `norm.cdf`이용
    - ex) norm.cdf(1.33): Z = 1.33까지의 누적확률

In [ ]:
from scipy.stats import norm
import numpy as np

# 데이터 설정
mu_0 = 8  # 모집단 평균
sigma = 1.5  # 모집단 표준편차
n = 100  # 표본 크기
x_bar = 8.2  # 표본 평균

# 표준오차 계산
SE = sigma / np.sqrt(n)

# Z-통계량 계산
z_stat = (x_bar - mu_0) / SE

# 양측 검정 p-value 계산
p_value = 2 * (1 - norm.cdf(abs(z_stat)))

# 결과 출력
print(f"Z-통계량: {z_stat:.2f}")
print(f"P-value: {p_value:.4f}")

# 유의수준 설정
alpha = 0.05
if p_value < alpha:
    print("귀무가설을 기각합니다. 새로운 정책이 만족도에 유의미한 영향을 미칩니다.")
else:
    print("귀무가설을 기각하지 못합니다. 새로운 정책이 만족도에 유의미한 영향을 미친다는 증거가 부족합니다.")

# 일표본 t검정

어떤 과자 공장에서 생산하는 A 과자의 중량은 평균 150g으로 알려져 있습니다. 품질 관리팀에서 최근 생산된 과자 30개를 무작위로 뽑아 무게를 측정했습니다. 과연 이 샘플의 평균 무게는 기존에 알려진 모평균 150g과 통계적으로 같다고 할 수 있을까요?

In [ ]:
# 모평균 설정
population_mean = 150

# 30개 과자 샘플의 무게 데이터
sample_data = [149.21, 151.59, 154.61, 149.04, 156.86,
               153.50, 155.22, 156.96, 148.39, 148.92,
               146.41, 155.81, 148.94, 150.27, 152.18,
               154.33, 155.64, 152.15, 150.54, 161.65,
               150.28, 141.55, 146.31, 153.71, 161.13,
               159.25, 145.71, 150.13, 151.87, 144.00]

In [ ]:
sample_mean = np.mean(sample_data)
sample_std = np.std(sample_data, ddof=1)
n = len(sample_data)
df = n - 1

print(f"가설 모평균 (μ₀): {population_mean}g")
print(f'측정된 과자 샘플의 평균 무게: {sample_mean:.2f}g')

In [ ]:
# 표준 오차 (Standard Error of the Mean, s / √n)
standard_error = sample_std / np.sqrt(n)
# t-통계량 공식 적용
t_statistic_manual = (sample_mean - population_mean) / standard_error

print(f"표준 오차 (SEM): {standard_error:.4f}")
print(f"t-통계량 (직접 계산): {t_statistic_manual:.4f}")

In [ ]:
# t-분포 그래프를 위한 데이터 생성
# -4에서 4까지의 범위를 400개로 나눔
x = np.linspace(-4, 4, 400)
# 해당 x값에 대한 t-분포의 확률 밀도 함수(PDF) 값 계산
y = stats.t.pdf(x, df)

# 그래프 그리기
plt.figure(figsize=(10, 6))
# t-분포 곡선 그리기
plt.plot(x, y, 'b-', label=f't-distribution (df={df})')
# t-통계량 위치에 수직선 그리기
plt.axvline(x=t_statistic_manual, color='r', linestyle='--', label=f't-statistic = {t_statistic_manual:.2f}')

# 유의수준 0.05 기각역 표시
alpha = 0.05
t_critical = stats.t.ppf(1 - alpha / 2, df)

x_fill_right = np.linspace(t_critical, 4, 100)
y_fill_right = stats.t.pdf(x_fill_right, df)
plt.fill_between(x_fill_right, y_fill_right, color='blue', alpha=0.5, label=f'Significance Level (a={alpha})')

x_fill_left = np.linspace(-4, -t_critical, 100)
y_fill_left = stats.t.pdf(x_fill_left, df)
plt.fill_between(x_fill_left, y_fill_left, color='blue', alpha=0.5)

plt.title('t-Distribution of Sample Data')
plt.xlabel('t-value')
plt.ylabel('Probability Density')
plt.legend()
plt.grid(True)

In [ ]:
# stats.t.sf() (Survival Function)는 1 - CDF 값으로, t-분포의 꼬리 한쪽 넓이를 계산합니다.
# 양측 검정이므로, 꼬리 넓이에 2를 곱해줍니다.
p_value_manual = stats.t.sf(np.abs(t_statistic_manual), df=df) * 2

print("--- p-값 계산 과정 ---")
print(f"p-값 (직접 계산): {p_value_manual:.4f}")

In [ ]:
alpha = 0.05
print(f"유의수준: {alpha}")
if p_value_manual < alpha:
    print(">> 결론: 귀무가설을 기각합니다. 샘플의 평균은 모평균과 유의미한 차이가 있습니다.")
else:
    print(">> 결론: 귀무가설을 기각할 수 없습니다.")

Scipy 라이브러리 사용해서 구현하기  
[scipy.stats.ttest_1samp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_1samp.html)

In [ ]:
t_statistic, p_value = stats.ttest_1samp(a=sample_data, popmean=population_mean)

print("--- 일표본 t-검정 결과 ---")
print(f"t-통계량 (t-statistic): {t_statistic:.4f}")
print(f"p-값 (p-value): {p_value:.4f}")

# 이표본 t검정

한 교육 연구원이 새로 개발한 수학 교육 방법(B)이 기존 교육 방법(A)보다 효과적인지 알아보고자 합니다. 이를 위해 50명의 학생을 무작위로 두 그룹(각 25명)으로 나누어, 한 그룹은 A 방식으로, 다른 그룹은 B 방식으로 교육한 후 시험을 치렀습니다. 과연 두 교육 방식 간에 시험 성적의 평균 차이가 통계적으로 유의미할까요?

In [ ]:
# 데이터 준비
group_a = np.array([85, 87, 92, 78, 83, 89, 91, 86, 84, 88, 90, 82, 87, 89, 85])
group_b = np.array([78, 81, 85, 74, 79, 83, 86, 80, 77, 82, 84, 76, 81, 83, 79])

In [ ]:
# 각 그룹의 표본 통계량 계산
mean_a = np.mean(group_a)
var_a = np.var(group_a, ddof=1)
n_a = len(group_a)

mean_b = np.mean(group_b)
var_b = np.var(group_b, ddof=1)
n_b = len(group_b)

print(f"그룹 A: Mean={mean_a:.2f}, Var={var_a:.2f}, n={n_a}")
print(f"그룹 B: Mean={mean_b:.2f}, Var={var_b:.2f}, n={n_b}")

In [ ]:
pooled_variance = ((n_a - 1) * var_a + (n_b - 1) * var_b) / (n_a + n_b - 2)
pooled_std = np.sqrt(pooled_variance)

t_statistic_manual = (mean_a - mean_b) / (pooled_std * np.sqrt(1/n_a + 1/n_b))

print(f"합동 분산 (Pooled Variance): {pooled_variance:.4f}")
print(f"t-통계량 (직접 계산): {t_statistic_manual:.4f}")

In [ ]:
degrees_of_freedom = n_a + n_b - 2
p_value_manual = stats.t.sf(np.abs(t_statistic_manual), df=degrees_of_freedom) * 2

print(f"자유도 (df): {degrees_of_freedom}")
print(f"p-값 (직접 계산): {p_value_manual:.4f}")

SciPy 라이브러리 사용해서 구현하기  
[scipy.stats.ttest_ind](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html#scipy.stats.ttest_ind)

In [ ]:
t_stat_lib, p_val_lib = stats.ttest_ind(group_a, group_b, equal_var=True)

print("--- SciPy 라이브러리 검증 결과 (Student's t-test) ---")
print(f"t-통계량 (라이브러리): {t_stat_lib:.4f}")
print(f"p-값 (라이브러리): {p_val_lib:.4f}")

# ANOVA 분산분석
한 농업 연구원이 새로 개발한 두 종류의 비료(B, C)가 식물 성장에 미치는 영향을 알아보고자 합니다. 비료를 주지 않은 통제 집단(A)을 포함하여 세 집단의 식물 키를 측정했습니다. 과연 세 집단의 평균 키에는 통계적으로 유의미한 차이가 있을까요?

In [ ]:
group_a = np.random.normal(20, 5, 20) # 통제 집단 (평균 20)
group_b = np.random.normal(28, 5, 20) # 비료 B (평균 28)
group_c = np.random.normal(25, 5, 20) # 비료 C (평균 25)

groups = [group_a, group_b, group_c]

In [ ]:
# 전체 데이터와 전체 평균(grand mean)
all_data = np.concatenate(groups)
grand_mean = np.mean(all_data)
N = len(all_data) # 전체 샘플 수
k = len(groups)   # 집단 수

In [ ]:
# --- 3. SSB (집단 간 제곱합) 계산 ---
ssb = sum([len(g) * (np.mean(g) - grand_mean)**2 for g in groups])

# --- 4. SSW (집단 내 제곱합) 계산 ---
ssw = sum([np.var(g, ddof=0) * len(g) for g in groups])

print(f"집단 간 제곱합 (SSB): {ssb:.4f}")
print(f"집단 내 제곱합 (SSW): {ssw:.4f}")

In [ ]:
df_between = k - 1
df_within = N - k

msb = ssb / df_between
msw = ssw / df_within

f_stat_manual = msb / msw

print(f"집단 간 자유도 (df_between): {df_between}")
print(f"집단 내 자유도 (df_within): {df_within}")
print(f"F-통계량 (직접 계산): {f_stat_manual:.4f}")

In [ ]:
p_value_manual = stats.f.sf(f_stat_manual, df_between, df_within)
print(f"p-값 (직접 계산): {p_value_manual:.4f}")

Scipy 라이브러리 사용해서 구현하기  
[scipy.stats.f_oneway](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html)

In [ ]:
f_stat_lib, p_val_lib = stats.f_oneway(group_a, group_b, group_c)

print("\n--- SciPy 라이브러리 검증 결과 ---")
print(f"F-통계량 (라이브러리): {f_stat_lib:.4f}")
print(f"p-값 (라이브러리): {p_val_lib:.4f}")

사후 분석하기  
tukey' HSD는 [statsmodels](https://www.statsmodels.org/stable/index.html) 라이브러리에서 지원합니다

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
all_data = np.concatenate([group_a, group_b, group_c])
group_labels = ['A'] * 20 + ['B'] * 20 + ['C'] * 20

df = pd.DataFrame({'value': all_data, 'group': group_labels})

# ---  Tukey's HSD 사후 분석 수행 ---
# pairwise_tukeyhsd(endog=종속변수, groups=그룹변수, alpha=유의수준)
tukey_result = pairwise_tukeyhsd(endog=df['value'], groups=df['group'], alpha=0.05)

print("--- Tukey's HSD 사후 분석 결과 ---")
print(tukey_result)

In [ ]:
tukey_result.plot_simultaneous()
plt.show()

# 카이제곱 재현
- 정규분포의 1,2,3,개의 합의 분포와 카이제곱 분포가 동일하다는 점을 보여봅시다.
- 정규 분포 자료 생성 함수는 `scipy.stats.norm.rvs` 입니다.
- 카이제곱 분포 생성 함수는 `scipy.stats.chi2.pdf` 입니다.

In [ ]:
#샘플 수 설정
n_samples = 10000

# Generate standard normal distributions
z1 = stats.norm.rvs(loc=0, scale=1, size=n_samples)
z2 = stats.norm.rvs(loc=0, scale=1, size=n_samples)
z3 = stats.norm.rvs(loc=0, scale=1, size=n_samples)

# Calculate chi-square distributions with 1, 2, and 3 degrees of freedom
chi2_1 = z1**2
chi2_2 = z1**2 + z2**2
chi2_3 = z1**2 + z2**2 + z3**2

# 그래프 크기와 x축 결정
plt.figure(figsize=(12, 6))
x = np.linspace(0, 15, 1000)
plt.xlim(0,15)

# 정규분포 데이터의 합으로 만든 카이제곱 그리기
plt.hist(chi2_1, bins=100, density=True, alpha=0.6, label='Chi-square (df=1)', color='blue')
plt.hist(chi2_2, bins=100, density=True, alpha=0.6, label='Chi-square (df=2)', color='green')
plt.hist(chi2_3, bins=100, density=True, alpha=0.6, label='Chi-square (df=3)', color='red')

# chi2.pdf 함수로 만든 이론적 카이제곱 그리기
plt.plot(x, stats.chi2.pdf(x, df=1), 'b-', label='Theoretical (df=1)')
plt.plot(x, stats.chi2.pdf(x, df=2), 'g-', label='Theoretical (df=2)')
plt.plot(x, stats.chi2.pdf(x, df=3), 'r-', label='Theoretical (df=3)')

# Labels and legend
plt.title('Chi-square Distribution for Different Degrees of Freedom')
plt.xlabel('Value')
plt.ylabel('Density')
plt.legend()
plt.grid()
plt.show()

**적합도 검정**

In [ ]:
# 주장하는 방문 비율(%)을 기반으로 기대값을 계산
expected_ratios = np.array([10, 10, 15, 20, 30, 15])
total_visits = 200
expected_counts = (expected_ratios / 100) * total_visits

# 관찰된 방문 수
observed_counts = np.array([30, 14, 34, 45, 57, 20])

# 카이제곱 적합도 검정을 수행
chi2_stat, p_val = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)

# 결과 출력
print(f"Chi-squared Statistic: {chi2_stat:.3f}") # 11.442
print(f"P-value: {p_val:.3f}") # 0.043

# 결론 도출
alpha = 0.05  # 유의수준
if p_val < alpha:
    print("귀무가설 기각: 관찰된 방문 비율은 주장하는 비율과 다르다.")
else:
    print("귀무가설 채택: 관찰된 방문 비율은 주장하는 비율과 같다.")

**독립성 검정**

In [ ]:
# 관찰된 데이터: 성별과 흡연 여부에 따른 빈도수
data = np.array([
    [40, 60],  # 남성: 흡연자, 비흡연자
    [30, 70]   # 여성: 흡연자, 비흡연자
])

# 카이제곱 통계량, p-value, 자유도, 기대값을 계산
chi2_stat, p_val, dof, expected = stats.chi2_contingency(data, correction=True)

# 결과 출력
print(f"Chi-squared Statistic: {chi2_stat:.3f}") #1.780
print(f"P-value: {p_val:.3f}") #0.182
print(f"Degrees of Freedom: {dof:.3f}") #1.000
print("Expected frequencies:")
print(expected)
'''
[[35. 65.]
 [35. 65.]]
 '''

# 결론
alpha = 0.05  # 유의수준
if p_val < alpha:
    print("귀무가설 기각: 성별과 흡연 여부는 독립적이지 않다.")
else:
    print("귀무가설 채택: 성별과 흡연 여부는 독립적이다.")

**동질성 검정**

In [ ]:
# 관찰된 데이터: 각 학교에서 학생들이 선호하는 과목의 빈도수
data = np.array([
    [50, 60, 55],  # 수학 선호
    [40, 45, 50],  # 과학 선호
    [30, 35, 40]   # 문학 선호
])

# 카이제곱 통계량, p-value, 자유도, 기대값을 계산
chi2_stat, p_val, dof, expected = stats.chi2_contingency(data)

# 결과 출력
print(f"Chi-squared Statistic: {chi2_stat:.3f}")# 0.817
print(f"P-value: {p_val:.3f}") # 0.936
print(f"Degrees of Freedom: {dof:.3f}") # 0.400
print("Expected frequencies:")
print(expected.round(3))

'''
[[48.889 57.037 59.074]
 [40.    46.667 48.333]
 [31.111 36.296 37.593]]
'''

# 결론
alpha = 0.05  # 유의수준
if p_val < alpha:
    print("귀무가설 기각: 각 학교에서 과목 선호도는 동일하지 않다.")
else:
    print("귀무가설 채택: 각 학교에서 과목 선호도는 동일하다.")

# 윌콕슨 순위합 검정(맨-휘트니 U 검정)

한 기업에서 두 가지 웹사이트 디자인(A, B)을 A/B 테스트한 후, 사용자들에게 만족도 점수(1~20점)를 받았습니다. 이 점수 데이터가 정규분포를 따른다고 확신할 수 없을 때, 두 디자인 간에 만족도 점수 분포에 차이가 있는지 맨-휘트니 U 검정으로 확인해 보겠습니다.

[scipy.stats.mannwhitneyu](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html#scipy.stats.mannwhitneyu)

In [ ]:
group_a = [7, 8, 9, 12, 15, 16, 17]
group_b = [1, 2, 3, 4, 5, 6, 10, 11]

u_stat_lib, p_val_lib = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')

print("--- SciPy 라이브러리 검증 결과 ---")
print(f"U 통계량: {u_stat_lib:.4f}")
print(f"p-값 (p-value): {p_val_lib:.4f}")

alpha = 0.05
if p_val_lib < alpha:
    print("\n결론: 귀무가설을 기각합니다. 두 디자인의 만족도 점수 분포는 유의미하게 다릅니다.")
else:
    print("\n결론: 귀무가설을 기각할 수 없습니다.")

---


# **문제 1 : 게임 아이템 뽑기 확률, 정말 10%일까?**

**시나리오:**
어떤 게임 개발사에서 새로운 아이템의 뽑기 확률이 10%로 설정되었다고 주장하고 있습니다. 유저 커뮤니티에서는 "실제 확률은 10%보다 낮은 것 같다"는 의혹이 제기되었습니다. 이를 확인하기 위해 한 유저가 아이템을 200번 뽑았고, 그중 12번 성공했습니다.


**Part 1: 시나리오 기반 문제 해결 (예시 답안 및 해설)**


In [ ]:
# 문제 설정
n = 200  # 총 시행 횟수
x = 12   # 관찰된 성공 횟수
p = 0.1  # 귀무가설에서의 성공 확률

# [작성] 가설을 주석으로 작성해보세요.
# H₀ (귀무가설): 아이템 뽑기 확률은 10%이다. (p = 0.1)
# H₁ (대립가설): 아이템 뽑기 확률은 10%보다 낮다. (p < 0.1)

# [작성] 이항검정을 수행하고 p-값을 계산하세요.
# '확률이 더 낮은 것 같다'는 주장을 검증하려면 alternative 인수를 어떻게 설정해야 할까요?
p_value = stats._______(k=x, n=n, p=p, alternative='less').pvalue



print(f"관찰 결과: {n}번 시도 중 {x}번 성공")
print(f"검정 결과 p-value: {p_value:.4f}")

# [작성] 유의수준 0.05를 기준으로 통계적 결론을 내리는 코드를 작성하세요.
alpha = 0.05
if p_value < alpha:
    print(f"결론: p-값이 유의수준({alpha})보다 작으므로 귀무가설을 기각합니다.")
    print("-> 즉, 아이템 뽑기 확률이 10%보다 낮다고 볼 통계적 근거가 있습니다.")
else:
    print(f"결론: p-값이 유의수준({alpha})보다 크거나 같으므로 귀무가설을 기각하지 못합니다.")
    print("-> 즉, 아이템 뽑기 확률이 10%보다 낮다고 단정할 수 없습니다.")

**Part 2: 시뮬레이션 기반 원리 탐구 (예시 답안 및 해설)**


In [ ]:
# 1. 시뮬레이션 설정
# H₀가 사실이라고 가정: 실제 확률 p = 0.1
# 200번 뽑기를 10,000번 반복 수행
num_simulations = 10000

# [작성] numpy의 이항분포 랜덤 함수를 사용하여 시뮬레이션 결과를 생성하세요.
simulated_successes = np.random.________(n=n, p=p, size=num_simulations)


# 2. 시각화
plt.figure(figsize=(12, 6))
sns.histplot(simulated_successes, discrete=True, stat='density', label='시뮬레이션 결과 분포\n(H₀가 사실일 때)')

plt.axvline(x=x, color='red', linestyle='--', label=f'실제 관찰값: {x}번 성공')
plt.title("'확률이 10%일 때' 200번 뽑기 시뮬레이션 (10,000회 반복)", fontsize=15)
plt.xlabel("성공 횟수", fontsize=12)
plt.ylabel("상대 빈도", fontsize=12)
plt.legend()
plt.show()

**🤔 종합 토의**

**1. Part 1에서 구한 `p-value`는 Part 2 그래프의 어느 부분에 해당하며, 무엇을 의미하나요?**

**2. 시뮬레이션 그래프를 볼 때, 우리가 관찰한 '12회 성공'은 개발사의 주장이 맞다는 가정 하에 흔한 일인가요, 드문 일인가요?**

**3. 이 시뮬레이션 경험을 통해 "p-값이 작으면 귀무가설을 기각한다"는 규칙을 친구에게 어떻게 더 쉽게 설명할 수 있을까요?**


<details>
<summary><strong>🧑‍🏫 해설 보기</strong></summary>

**🤔 종합 토의 (해설 및 답변 예시)**

**1. Part 1에서 구한 `p-value`는 Part 2 그래프의 어느 부분에 해당하며, 무엇을 의미하나요?**

- **답변:** Part 1에서 구한 p-값(약 0.035)은 **"귀무가설이 사실일 때(확률이 10%일 때), 우리가 관찰한 값(12) 또는 그보다 더 극단적인 값(12 이하)이 나올 확률"**을 의미합니다. 이는 Part 2 그래프에서 **빨간 점선(x=12)을 포함한 왼쪽 꼬리 부분의 전체 면적**에 해당합니다.

**2. 시뮬레이션 그래프를 볼 때, 우리가 관찰한 '12회 성공'은 개발사의 주장이 맞다는 가정 하에 흔한 일인가요, 드문 일인가요?**

- **답변:** 그래프의 중심(가장 빈도가 높은 곳)은 20회 근처입니다. 반면 우리가 관찰한 12회는 분포의 중심에서 멀리 떨어진 **왼쪽 꼬리 부분에 위치**하므로, **매우 드물게(Unlikely) 발생하는 일**임을 알 수 있습니다.

**3. 이 시뮬레이션 경험을 통해 "p-값이 작으면 귀무가설을 기각한다"는 규칙을 친구에게 어떻게 더 쉽게 설명할 수 있을까요?**

- **답변 예시:** "네 말이 맞다고 치고(귀무가설) 시뮬레이션을 수천 번 돌려봤는데, 내가 겪은 일(관찰값)은 거의 일어나지 않는 희귀한 현상이더라. 그렇다면 '이건 우연이 아니라, 네가 처음에 말한 '네 말이 맞다'는 주장이 틀렸을 가능성이 높지 않을까?'라고 결론 내리는 거야. 즉, **내 주장이 맞다고 가정했을 때 너무 희귀한 일이 벌어지면, 처음 가정을 의심하는 거지.**"


---


#**문제 2 : 과자 한 봉지의 중량은 150g이 맞을까?**

**시나리오:**
한 식품 공장에서 생산하는 과자 한 봉지의 목표 중량은 150g입니다. 품질관리팀은 생산 공정이 목표 중량을 잘 맞추고 있는지 확인하기 위해, 생산된 과자 30봉지를 무작위로 추출하여 무게를 측정했습니다. (측정 데이터는 아래 코드에 제공)


**Part 1: 시나리오 기반 문제 해결 (예시 답안 및 해설)**


In [ ]:
# 주어진 데이터
np.random.seed(42)
sample_weights = np.random.normal(loc=151.5, scale=2, size=30)
pop_mean = 150 # 목표 중량 (모평균)

# [작성] 가설을 주석으로 작성해보세요.
# H₀ (귀무가설): 과자 봉지의 평균 중량은 150g이다.
# H₁ (대립가설): 과자 봉지의 평균 중량은 150g이 아니다.

# [작성] 일표본 t-검정을 수행하여 t-통계량과 p-값을 구하세요.
t_statistic, p_value = stats.______(a=sample_weights, popmean=pop_mean)


print(f"샘플 평균 무게: {np.mean(sample_weights):.2f}g")
print(f"t-statistic: {t_statistic:.4f}")
print(f"p-value: {p_value:.4f}")

# [작성] 유의수준 0.05를 기준으로 통계적 결론을 내리는 코드를 작성하세요.
alpha = ____
if p_value < alpha:
    print(f"결론: p-값이 유의수준({alpha})보다 작으므로 귀무가설을 기각합니다.")
    print("-> 즉, 과자 봉지의 평균 중량은 150g과 통계적으로 유의미한 차이가 있습니다.")
else:
    print(f"결론: p-값이 유의수준({alpha})보다 크거나 같으므로 귀무가설을 기각하지 못합니다.")
    print("-> 즉, 과자 봉지의 평균 중량이 150g과 다르다고 말할 통계적 근거가 부족합니다.")

**Part 2: 시뮬레이션 기반 원리 탐구 (예시 답안 및 해설)**


In [ ]:
# 1. 시뮬레이션 설정
# H₀가 사실이라고 가정: 평균(loc) = 150g
# 표준편차는 우리가 가진 샘플의 표준편차를 사용한다고 가정
# 30개짜리 샘플을 10,000번 뽑기
num_simulations = 10000
sample_size = 30
simulated_means = []

# [작성] for 반복문을 사용하여 시뮬레이션을 10,000번 수행하세요.
# 각 반복마다 H₀가 사실인 모집단에서 30개의 샘플을 뽑고, 그 평균을 simulated_means 리스트에 추가하세요.
for _ in range(num_simulations):
    # H₀가 사실인 모집단(평균=150)에서 샘플링
    simulated_sample = np.random.normal(loc=pop_mean, scale=np.std(sample_weights, ddof=1), size=sample_size)
    simulated_means.append(np.mean(simulated_sample))

# --- 해설 ---
# 귀무가설이 사실인 상황을 가정하기 위해, 평균(loc)은 150으로 설정합니다.
# 모집단의 표준편차는 알 수 없으므로, 우리가 가진 샘플의 표준편차(표본표준편차, ddof=1)를 추정치로 사용합니다.
# 이 가상 모집단에서 30개짜리 샘플을 10,000번 뽑아 각 샘플의 평균을 저장합니다.

# 2. 시각화
plt.figure(figsize=(12, 6))
sns.histplot(simulated_means, kde=True, label='시뮬레이션으로 얻은 표본 평균들의 분포\n(H₀가 사실일 때)')
plt.axvline(x=np.mean(sample_weights), color='red', linestyle='--', label=f'실제 관찰된 표본 평균: {np.mean(sample_weights):.2f}g')
plt.title("표본 평균의 분포 시뮬레이션", fontsize=15)
plt.xlabel("표본 평균 (g)", fontsize=12)
plt.ylabel("빈도", fontsize=12)
plt.legend()
plt.show()

**🤔 종합 토의**


**1. Part 2의 히스토그램은 무엇을 나타내나요? 이 분포의 모양이 정규분포와 비슷한 이유는 무엇일까요? (힌트: 4장에서 배운 개념)**



**2. 우리가 실제로 관찰한 표본 평균(빨간 점선)은 이 분포에서 흔하게 나타나는 값인가요? Part 1의 p-값과 이 시각적 위치는 어떻게 관련되나요?**



**3. 만약 p-값이 0.001이었다면, 빨간 점선은 그래프의 어디쯤에 위치할 것으로 예상되나요? 이는 공장 입장에서 어떤 조치를 취해야 함을 시사할까요?**



<details>
<summary><strong>🧑‍🏫 해설 보기</strong></summary>

**🤔 종합 토의 (해설 및 답변 예시)**

**1. Part 2의 히스토그램은 무엇을 나타내나요? 이 분포의 모양이 정규분포와 비슷한 이유는 무엇일까요? (힌트: 4장에서 배운 개념)**

- **답변:** 이 히스토그램은 **'표본 평균의 표집 분포(Sampling Distribution of the Sample Mean)'**를 시뮬레이션으로 구현한 것입니다. 모집단의 분포와 상관없이 표본의 크기(n=30)가 충분히 크면, 표본 평균들의 분포는 정규분포에 가까워집니다. 이를 **중심극한정리(Central Limit Theorem, CLT)**라고 합니다.

**2. 우리가 실제로 관찰한 표본 평균(빨간 점선)은 이 분포에서 흔하게 나타나는 값인가요? Part 1의 p-값과 이 시각적 위치는 어떻게 관련되나요?**

- **답변:** 빨간 점선은 분포의 중심에서 매우 멀리 떨어진 **오른쪽 꼬리**에 위치합니다. 이는 귀무가설이 사실일 때 매우 드물게 나타나는 값임을 의미합니다. Part 1의 p-값(약 0.0001)은 이 빨간 점선보다 더 극단적인 값(오른쪽 꼬리 전체와, 대칭적인 왼쪽 꼬리 부분)이 나타날 확률을 의미하며, 이 값이 매우 작다는 것은 시각적 판단과 일치합니다.

**3. 만약 p-값이 0.001이었다면, 빨간 점선은 그래프의 어디쯤에 위치할 것으로 예상되나요? 이는 공장 입장에서 어떤 조치를 취해야 함을 시사할까요?**

- **답변:** p-값이 0.001이라도 여전히 매우 작은 값이므로, 빨간 점선은 분포의 중심에서 멀리 떨어진 꼬리 부분에 위치할 것입니다. 이는 생산 공정에 문제가 있어 과자 중량이 목표치(150g)와 체계적으로 다르게 생산되고 있음을 강력히 시사합니다. 공장 입장에서는 즉시 **생산 설비를 점검하고 공정을 재조정**하는 등의 조치를 취해야 합니다.


---


# **문제 3 : 어떤 온라인 학습 방식이 더 효과적일까?**

**시나리오:**
한 교육 기업에서 두 가지 다른 온라인 학습 방식(A, B)을 개발했습니다. 방식 A가 방식 B보다 학생들의 성적 향상에 더 효과적인지 알아보기 위해, 두 그룹의 학생들에게 각각 다른 방식으로 한 달간 학습시킨 후 시험을 보게 했습니다.


**Part 1: 시나리오 기반 문제 해결 (예시 답안 및 해설)**


In [ ]:
# 데이터 생성
np.random.seed(0)
group_a_scores = np.random.normal(loc=85, scale=8, size=50)
group_b_scores = np.random.normal(loc=80, scale=7, size=50)

# [작성] 1. 정규성 검정 (Shapiro-Wilk test)
# H₀: 데이터는 정규분포를 따른다.
# 두 그룹에 대해 각각 정규성 검정을 수행하고 p-값을 출력하세요.
shapiro_a_pvalue = stats.______(group_a_scores).pvalue
shapiro_b_pvalue = stats.______(group_b_scores).pvalue
print(f"A그룹 정규성 검정 p-value: {shapiro_a_pvalue:.4f}")
print(f"B그룹 정규성 검정 p-value: {shapiro_b_pvalue:.4f}")


# [작성] 2. 등분산성 검정 (Levene's test)
# H₀: 두 그룹의 분산은 같다.
# 두 그룹에 대해 등분산성 검정을 수행하고 p-값을 출력하세요.
levene_pvalue = stats._____(group_a_scores, group_b_scores).pvalue
print(f"등분산성 검정 p-value: {levene_pvalue:.4f}")



# [작성] 3. 이표본 t-검정
# H₀: 두 그룹의 평균은 같다.
# H₁: 두 그룹의 평균은 다르다.
# 등분산성 가정을 만족했는지 여부에 따라 equal_var 인수를 설정하여 t-검정을 수행하세요.
t_statistic, p_value = stats.______(a=group_a_scores, b=group_b_scores, equal_var=True)
print(f"\n이표본 t-검정 t-statistic: {t_statistic:.4f}")
print(f"이표본 t-검정 p-value: {p_value:.4f}")



**Part 2: 시뮬레이션 기반 원리 탐구 (예시 답안 및 해설)**


In [ ]:
# 1. 시뮬레이션 설정
# H₀가 사실이라고 가정: 두 그룹의 평균은 같다.
# 두 그룹의 모든 데이터를 합쳐서 하나의 거대한 모집단(H₀)을 만듭니다.
combined_scores = np.concatenate([group_a_scores, group_b_scores])
num_simulations = 10000
simulated_diffs = []

# [작성] for 반복문을 사용하여 시뮬레이션을 10,000번 수행하세요.
# 각 반복마다, 합쳐진 데이터(combined_scores)에서 비복원추출로 50개(가상 A그룹)와 나머지 50개(가상 B그룹)를 뽑아
# 두 그룹의 평균 차이를 계산하고 simulated_diffs 리스트에 추가하세요.
for _ in range(num_simulations):
    # 데이터를 무작위로 섞은 후, 앞의 50개와 뒤의 50개로 나눔
    np.random._______(combined_scores)
    sim_group_a = combined_scores[:50]
    sim_group_b = combined_scores[50:]
    sim_diff = np.mean(sim_group_a) - np.mean(sim_group_b)
    simulated_diffs.append(sim_diff)



# 2. 시각화
observed_diff = np.mean(group_a_scores) - np.mean(group_b_scores)
plt.figure(figsize=(12, 6))
sns.histplot(simulated_diffs, kde=True, label='시뮬레이션으로 얻은 평균 차이의 분포\n(H₀가 사실일 때)')
plt.axvline(x=observed_diff, color='red', linestyle='--', label=f'실제 관찰된 평균 차이: {observed_diff:.2f}')
plt.title("두 그룹 간 평균 차이의 분포 시뮬레이션", fontsize=15)
plt.xlabel("A그룹 평균 - B그룹 평균", fontsize=12)
plt.ylabel("빈도", fontsize=12)
plt.legend()
plt.show()

**🤔 종합 토의**

**1. 정규성 검정과 등분산성 검정의 p-값은 각각 어떻게 해석해야 하나요? 이 결과는 우리가 이표본 t-검정을 사용하는 데 문제가 없음을 보여주나요?**


**2. Part 2의 히스토그램 중심이 0에 가까운 이유는 무엇일까요?**


**3. 실제 관찰된 평균 차이(빨간 점선)는 "두 방식의 효과가 같다"고 가정했을 때 우연히 나타날 수 있는 범위 안에 있나요, 아니면 그 범위를 벗어나나요? 이 시각적 판단과 Part 1의 p-값은 어떤 관계가 있나요?**


<details>
<summary><strong>🧑‍🏫 해설 보기</strong></summary>

**🤔 종합 토의 (해설 및 답변 예시)**

**1. 정규성 검정과 등분산성 검정의 p-값은 각각 어떻게 해석해야 하나요? 이 결과는 우리가 이표본 t-검정을 사용하는 데 문제가 없음을 보여주나요?**

- **답변:** 정규성 검정 p-값(0.45, 0.87)과 등분산성 검정 p-값(0.55) 모두 유의수준 0.05보다 큽니다. 이는 "데이터가 정규분포를 따른다"는 귀무가설과 "두 그룹의 분산이 같다"는 귀무가설을 기각할 수 없음을 의미합니다. 따라서 t-검정의 기본 가정을 모두 만족하므로, 이표본 t-검정을 사용하기에 적절합니다.

**2. Part 2의 히스토그램 중심이 0에 가까운 이유는 무엇일까요?**

- **답변:** 이 히스토그램은 "두 그룹의 효과가 같다"는 가정 하에 만들어졌습니다. 두 그룹이 같다면, 무작위로 데이터를 섞어 두 그룹으로 나누었을 때 두 그룹의 평균 차이는 0에 가까울 확률이 가장 높습니다. 따라서 분포의 중심은 0이 됩니다.

**3. 실제 관찰된 평균 차이(빨간 점선)는 "두 방식의 효과가 같다"고 가정했을 때 우연히 나타날 수 있는 범위 안에 있나요, 아니면 그 범위를 벗어나나요? 이 시각적 판단과 Part 1의 p-값은 어떤 관계가 있나요?**

- **답변:** 실제 관찰된 평균 차이(약 3.99)는 시뮬레이션 분포의 중심에서 매우 멀리 떨어진 **오른쪽 꼬리**에 위치합니다. 이는 우연히 발생하기 매우 어려운 극단적인 값임을 시각적으로 보여줍니다. Part 1의 p-값(0.004)은 이 빨간 점선보다 더 극단적인 값들이 나타날 확률이며, 이 값이 매우 작다는 것은 시각적 판단과 정확히 일치합니다.


---


# **문제 4 : 연령대별로 선호하는 영화 장르가 다를까?**

**시나리오:**
한 영화관에서 고객의 연령대(20대, 30대, 40대)에 따라 선호하는 영화 장르(액션, 로맨스)에 차이가 있는지 궁금해졌습니다. 이를 알아보기 위해 300명의 고객을 대상으로 설문조사를 진행하여 아래와 같은 분할표를 얻었습니다.


**Part 1: 시나리오 기반 문제 해결 (예시 답안 및 해설)**


In [ ]:
import pandas as pd

# 데이터 생성 (관측 빈도 분할표)
data = {'액션': [70, 50, 30],
        '로맨스': [30, 60, 60]}
observed = pd.DataFrame(data, index=['20대', '30대', '40대'])
print("관측 빈도 (Observed Frequencies):")
print(observed)

# [작성] 가설을 주석으로 작성해보세요.
# H₀ (귀무가설): 연령대와 선호 장르는 서로 독립이다 (관련이 없다).
# H₁ (대립가설): 연령대와 선호 장르는 서로 독립이 아니다 (관련이 있다).

# [작성] 카이제곱 독립성 검정을 수행하고 카이제곱 통계량, p-값, 기대 빈도를 구하세요.
chi2, p_value, dof, expected = stats.__________(observed)


print(f"\nChi-squared statistic: {chi2:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Degrees of Freedom: {dof}")
print("\n기대 빈도 (Expected Frequencies):")
print(pd.DataFrame(expected, index=observed.index, columns=observed.columns))

**Part 2: 시뮬레이션 기반 원리 탐구 (예시 답안 및 해설)**


In [ ]:
# 1. 시뮬레이션 설정
# H₀가 사실이라고 가정: 두 변수는 독립이다.
# 전체 비율을 기반으로 가상 데이터를 생성합니다.
total_people = observed.sum().sum()
p_genre = observed.sum(axis=0) / total_people # 장르별 전체 비율
p_age = observed.sum(axis=1) / total_people   # 연령대별 전체 비율
num_simulations = 5000
simulated_chi2_stats = []

# [작성] for 반복문을 사용하여 시뮬레이션을 5,000번 수행하세요.
# 각 반복마다, 300명의 가상 고객에게 연령대와 장르를 '독립적으로' 할당한 후,
# 가상 분할표를 만들고 카이제곱 통계량을 계산하여 simulated_chi2_stats 리스트에 추가하세요.
for _ in range(num_simulations):
    # 300명의 가상 연령대와 장르를 독립적으로 생성
    sim_ages = np.random.choice(p_age.index, size=total_people, p=p_age.values)
    sim_genres = np.random.choice(p_genre.index, size=total_people, p=p_genre.values)

    # 가상 분할표 생성
    sim_crosstab = pd.crosstab(sim_ages, sim_genres)

    # 카이제곱 통계량 계산
    # 생성된 crosstab에 모든 카테고리가 포함되지 않을 수 있어, 원래 observed의 구조에 맞춰 재정렬
    sim_crosstab = sim_crosstab.reindex(index=observed.index, columns=observed.columns, fill_value=0)
    sim_chi2 = stats.___________(sim_crosstab)[0]
    simulated_chi2_stats.append(sim_chi2)


# 2. 시각화
plt.figure(figsize=(12, 6))
sns.histplot(simulated_chi2_stats, kde=True, label='시뮬레이션으로 얻은 카이제곱 통계량 분포\n(H₀가 사실일 때)')
plt.axvline(x=chi2, color='red', linestyle='--', label=f'실제 관찰된 카이제곱 통계량: {chi2:.2f}')
plt.title("카이제곱 통계량의 분포 시뮬레이션", fontsize=15)
plt.xlabel("카이제곱 통계량", fontsize=12)
plt.ylabel("빈도", fontsize=12)
plt.legend()
plt.show()

**🤔 종합 토의 (해설 및 답변 예시)**

**1. '기대 빈도'는 어떤 의미를 가지며, '관측 빈도'와의 차이가 클수록 카이제곱 통계량은 어떻게 변할까요?**

**2. Part 2의 히스토그램은 어떤 분포를 시각화한 것인가요? 실제 관찰된 카이제곱 통계량(빨간 점선)은 이 분포에서 흔한 값인가요, 아니면 극단적인 값인가요?**


**3. 유의수준 0.05에서, 연령대와 선호하는 영화 장르 사이에 통계적으로 유의미한 연관성이 있다고 결론 내릴 수 있습니까? 그 이유는 무엇인가요?**


<details>
<summary><strong>🧑‍🏫 해설 보기</strong></summary>

**🤔 종합 토의 (해설 및 답변 예시)**

**1. '기대 빈도'는 어떤 의미를 가지며, '관측 빈도'와의 차이가 클수록 카이제곱 통계량은 어떻게 변할까요?**

- **답변:** '기대 빈도'는 "만약 연령대와 선호 장르가 아무 관련이 없다면(독립이라면), 각 셀에 몇 명이 있을 것으로 기대되는가?"를 계산한 값입니다. '관측 빈도'와 '기대 빈도'의 차이가 클수록, 두 변수가 관련이 없다는 가정이 현실과 맞지 않다는 증거가 되며, 카이제곱 통계량 값은 커집니다.

**2. Part 2의 히스토그램은 어떤 분포를 시각화한 것인가요? 실제 관찰된 카이제곱 통계량(빨간 점선)은 이 분포에서 흔한 값인가요, 아니면 극단적인 값인가요?**

- **답변:** 이 히스토그램은 자유도가 2인 **카이제곱 분포(Chi-squared distribution)**를 시각화한 것입니다. 실제 관찰된 카이제곱 통계량(22.26)은 이 분포에서 우연히 나타나기에는 거의 불가능한, 매우 극단적인 위치(오른쪽 꼬리)에 있습니다.

**3. 유의수준 0.05에서, 연령대와 선호하는 영화 장르 사이에 통계적으로 유의미한 연관성이 있다고 결론 내릴 수 있습니까? 그 이유는 무엇인가요?**

- **답변:** 네, 결론 내릴 수 있습니다. 그 이유는 카이제곱 검정의 p-값(0.000015)이 유의수준 0.05보다 훨씬 작기 때문입니다. 이는 귀무가설("두 변수는 관련이 없다")을 기각하고 대립가설("두 변수는 관련이 있다")을 채택할 충분한 통계적 근거가 됨을 의미합니다.


---


#**문제 5 : 어떤 신규 비료가 가장 효과적일까?**

**시나리오:**
한 농업 연구소에서 새로 개발한 비료 3종류(A, B, C)의 생산량 증대 효과를 비교하고자 합니다. 동일한 조건의 밭 30개를 준비하여, 각각 10개씩 비료 A, B, C를 투여한 후 수확량을 측정했습니다.


**Part 1: 시나리오 기반 문제 해결 (예시 답안 및 해설)**


In [ ]:
# 데이터 생성
np.random.seed(123)
fertilizer_a = np.random.normal(loc=105, scale=10, size=10)
fertilizer_b = np.random.normal(loc=120, scale=12, size=10)
fertilizer_c = np.random.normal(loc=108, scale=9, size=10)

# 데이터프레임으로 변환 (사후분석을 위해)
df = pd.DataFrame({'수확량': np.concatenate([fertilizer_a, fertilizer_b, fertilizer_c]),
                   '비료종류': ['A']*10 + ['B']*10 + ['C']*10})

# 1. 시각화 (데이터 탐색)
sns.boxplot(x='비료종류', y='수확량', data=df)
plt.title('비료 종류에 따른 수확량 분포')
plt.show()

# [작성] 2. 분산분석 (ANOVA)
# H₀: 세 비료의 평균 수확량은 모두 같다.
# H₁: 적어도 하나 이상의 비료 평균 수확량은 다르다.
# 세 그룹에 대해 ANOVA를 수행하고 F-통계량과 p-값을 구하세요.
f_statistic, p_value = stats._______(fertilizer_a, fertilizer_b, fertilizer_c)
print(f"ANOVA F-statistic: {f_statistic:.4f}")
print(f"ANOVA p-value: {p_value:.4f}")


# [작성] 3. 사후분석 (Tukey's HSD)
# ANOVA의 p-값이 유의수준 0.05보다 작을 경우에만 사후분석을 수행하는 코드를 작성하세요.
if p_value < 0.05:
    # 여기에 Tukey's HSD 검정 코드를 작성하고 결과를 출력하세요.
    tukey_result = ____________(endog=df['수확량'], groups=df['비료종류'], alpha=0.05)
    print("\nTukey's HSD 사후분석 결과:")
    print(tukey_result)
else:
    print("\nANOVA 결과가 유의미하지 않으므로 사후분석을 수행하지 않습니다.")


**Part 2: 시뮬레이션 기반 원리 탐구 (예시 답안 및 해설)**


In [ ]:
# 1. 시뮬레이션 설정
# H₀가 사실이라고 가정: 세 그룹의 평균은 모두 같다.
# 모든 데이터를 합쳐서 하나의 거대한 모집단(H₀)을 만들고, 그 평균과 표준편차를 구합니다.
combined_harvest = df['수확량']
grand_mean = combined_harvest.mean()
grand_std = combined_harvest.std()
num_simulations = 5000
simulated_f_stats = []

# [작성] for 반복문을 사용하여 시뮬레이션을 5,000번 수행하세요.
# 각 반복마다, H₀ 모집단에서 10개씩 3개의 가상 그룹을 생성하고,
# 이 가상 그룹들로 ANOVA를 수행하여 F-통계량을 계산하여 simulated_f_stats 리스트에 추가하세요.
for _ in range(num_simulations):
    # H₀가 사실인 모집단에서 3개의 가상 그룹을 샘플링
    sim_group_a = np.random.normal(loc=grand_mean, scale=grand_std, size=10)
    sim_group_b = np.random.normal(loc=grand_mean, scale=grand_std, size=10)
    sim_group_c = np.random.normal(loc=grand_mean, scale=grand_std, size=10)

    # 가상 그룹들로 F-통계량 계산
    sim_f, _ = stats._______(sim_group_a, sim_group_b, sim_group_c)
    simulated_f_stats.append(sim_f)


# 2. 시각화
plt.figure(figsize=(12, 6))
sns.histplot(simulated_f_stats, kde=True, label='시뮬레이션으로 얻은 F-통계량 분포\n(H₀가 사실일 때)')
plt.axvline(x=f_statistic, color='red', linestyle='--', label=f'실제 관찰된 F-통계량: {f_statistic:.2f}')
plt.title("F-통계량의 분포 시뮬레이션", fontsize=15)
plt.xlabel("F-통계량", fontsize=12)
plt.ylabel("빈도", fontsize=12)
plt.legend()
plt.show()

**🤔 종합 토의**

**1. ANOVA 검정의 p-값을 통해 어떤 결론을 내릴 수 있나요? 이 결과는 "세 비료가 모두 동일한 효과를 가진다"는 것을 의미하나요, 아니면 "적어도 하나는 다르다"는 것을 의미하나요?**


**2. Tukey's HSD 결과표의 `reject` 열을 보세요. `True`로 표시된 조합은 무엇이며, 이는 무엇을 의미하나요? `meandiff` 열의 값은 어떤 정보를 주나요?**


**3. 이 모든 분석 결과를 종합하여, 연구소에 어떤 비료를 추천하고 그 이유는 무엇인지 비즈니스 관점에서 설명해 보세요.**


<details>
<summary><strong>🧑‍🏫 해설 보기</strong></summary>

**🤔 종합 토의 (해설 및 답변 예시)**

**1. ANOVA 검정의 p-값을 통해 어떤 결론을 내릴 수 있나요? 이 결과는 "세 비료가 모두 동일한 효과를 가진다"는 것을 의미하나요, 아니면 "적어도 하나는 다르다"는 것을 의미하나요?**

- **답변:** ANOVA의 p-값(0.013)은 유의수준 0.05보다 작으므로 귀무가설("세 비료의 평균 수확량은 모두 같다")을 기각합니다. 이는 "세 비료 중 **적어도 하나는** 다른 비료와 평균 수확량에서 유의미한 차이를 보인다"는 것을 의미합니다. 모든 비료가 다르다는 뜻은 아닙니다.

**2. Tukey's HSD 결과표의 `reject` 열을 보세요. `True`로 표시된 조합은 무엇이며, 이는 무엇을 의미하나요? `meandiff` 열의 값은 어떤 정보를 주나요?**

- **답변:** `reject` 열에서 `True`로 표시된 조합은 'A'와 'B' 입니다. 이는 **비료 A와 비료 B의 평균 수확량 간에 통계적으로 유의미한 차이가 있음**을 의미합니다. A-C, B-C 조합은 `False`이므로 유의미한 차이가 없다고 봅니다. `meandiff` 열은 각 그룹 쌍의 평균 차이를 보여줍니다. 예를 들어, A-B의 `meandiff`가 -14.28이라는 것은 B의 평균 수확량이 A보다 약 14.28만큼 높다는 의미입니다.

**3. 이 모든 분석 결과를 종합하여, 연구소에 어떤 비료를 추천하고 그 이유는 무엇인지 비즈니스 관점에서 설명해 보세요.**

- **답변 예시:** "분석 결과, 비료 A, B, C 간에는 유의미한 수확량 차이가 존재하며, 구체적으로 **비료 B가 비료 A보다 유의미하게 높은 수확량**을 보였습니다. 비료 B와 C 간에는 통계적으로 유의미한 차이가 발견되지 않았지만, 평균 수확량 자체는 B가 가장 높았습니다. 따라서 **가장 높은 생산성을 기대할 수 있는 비료 B의 도입을 추천**합니다. 다만, 비료 C도 A보다는 높은 수확량을 보이는 경향이 있고 B와 큰 차이가 없으므로, 만약 B의 가격이 C보다 월등히 비싸다면 **비용 대비 효율을 고려하여 C를 선택하는 것도 합리적인 대안**이 될 수 있습니다."
